<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W1_SQL_Zomato%20Case.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the Kaggle CLI
!pip install kaggle --quiet

# 2. Upload your Kaggle API token
#    • On Kaggle: Account → Create New API Token → download kaggle.json
#    • In Colab:
from google.colab import files
files.upload()   # select your kaggle.json

# 3. Configure the CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download & unzip the dataset
!kaggle datasets download -d equilibriumm/sleep-efficiency
!unzip -q sleep-efficiency.zip   # adjust if the zip has a folder

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency
License(s): copyright-authors


In [1]:

# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


In [24]:
users = pd.read_excel('/content/zomato_users.xlsx')
food = pd.read_excel('/content/zomato_food.xlsx')
menu = pd.read_excel('/content/zomato_menu.xlsx')
orders = pd.read_excel('/content/zomato_orders.xlsx')
order_detail = pd.read_excel('/content/zomato_order_detail.xlsx')
partner = pd.read_excel('/content/zomato_partner.xlsx')
res= pd.read_excel('/content/zomato_res.xlsx')

# Data Details

In [4]:
users.head(2)

,user_id,name,email,password
0,1,Nitish,nitish@gmail.com,p252h
1,2,Khushboo,khushboo@gmail.com,hxn9b


In [5]:
food.head(2)

,f_id,f_name,type
0,1,Non-veg Pizza,Non-veg
1,2,Veg Pizza,Veg


In [6]:
menu.head(2)

,menu_id,r_id,f_id,price
0,1,1,1,450
1,2,1,2,400


In [25]:
orders.head(2)

,order_id,user_id,r_id,amount,date,partner_id,delivery_time,delivery_rating,restaurant_rating
0,1001,1,1,550,2022-05-10,1,25,5,3.0
1,1002,1,2,415,2022-05-26,1,19,5,2.0


In [8]:
order_detail.head(2)

,id,order_id,f_id
0,1,1001,1
1,2,1001,3


In [9]:
partner.head(2)

,partner_id,partner_name
0,1,Suresh
1,2,Amit


In [10]:
res.head(2)

,r_id,r_name,cuisine
0,1,dominos,Italian
1,2,kfc,American


# Q1: Find the number of menu of restaurants

In [18]:
pysqldf('''
SELECT res.r_name,COUNT(menu.menu_id)
FROM res
INNER JOIN menu ON res.r_id = menu.r_id
GROUP BY res.r_name
''')

,r_name,COUNT(menu.menu_id)
0,China Town,3
1,Dosa Plaza,3
2,box8,3
3,dominos,3
4,kfc,3


# Q2 Find number of votes and avg rating of all restaurants

In [27]:
pysqldf(
    '''
    SELECT res.r_name,COUNT(o.restaurant_rating),AVG(o.restaurant_rating)
    FROM res
    INNER JOIN orders AS o ON res.r_id = o.r_id
    WHERE o.restaurant_rating IS NOT NULL
    GROUP BY res.r_name

    '''


)

,r_name,COUNT(o.restaurant_rating),AVG(o.restaurant_rating)
0,China Town,3,3.666667
1,Dosa Plaza,3,3.666667
2,box8,3,4.666667
3,dominos,3,1.666667
4,kfc,5,2.200000


# Q3 FInd the food that is being sold at most number of restaurants